# Legal RAG: evaluating exact supporting text

**Kevin · October 20–26**

Research question: can we evaluate the same evidence target when retrieval chunks change size?

This notebook presents a 20-question development pilot, 27 proposed supporting spans, and actual BM25 experiments. **Offsets are source-verified; evidence selections are assistant-reviewed drafts pending human approval.**

## 1. Source and scope

Reuse the existing reconstructed parent texts and child mappings. The sources are not original PDF coordinates. No new legal content is generated, and the 80 held-out questions are not used.

All results are saved. The run flag below is off by default; no GPU or API is required.

In [1]:
from pathlib import Path
import csv, json, html, sys
from IPython.display import display, HTML

ROOT = Path.cwd()
if not (ROOT / 'span_bench.py').exists():
    ROOT = ROOT / 'legal_rag_spans'
assert (ROOT / 'span_bench.py').exists(), 'Open the notebook from legal_rag_spans or its parent.'
sys.path.insert(0, str(ROOT))
import span_bench as bench

def table(rows, columns=None):
    rows = list(rows)
    if not rows:
        print('No rows.')
        return
    columns = columns or list(rows[0])
    header = ''.join('<th>' + html.escape(str(c)) + '</th>' for c in columns)
    body = ''.join('<tr>' + ''.join('<td>' + html.escape(str(r.get(c, ''))) + '</td>' for c in columns) + '</tr>' for r in rows)
    display(HTML('<table><thead><tr>' + header + '</tr></thead><tbody>' + body + '</tbody></table>'))

def csv_rows(name):
    with (ROOT / 'results' / name).open(encoding='utf-8') as f:
        return list(csv.DictReader(f))

documents, annotations, tokenizer = bench.load(ROOT)
table([{'Questions': len(annotations), 'Proposed spans': sum(len(a['spans']) for a in annotations),
        'Canonical parents': len(documents), 'Human approved': sum(a['review_status']=='human_approved' for a in annotations)}])


Questions,Proposed spans,Canonical parents,Human approved
20,27,2785,0


## 2. What an exact span means

A span stores the source ID, source hash, start/end character positions and exact supporting text. Coordinates are zero-based and half-open: `source_text[start:end]` must equal the evidence.

Question 95 asks for both conduct and mental state. Its proposed evidence set has two clauses in **one** original passage—not a newly validated multi-document question.

In [2]:
example = next(a for a in annotations if a['question_id'] == '95')
print('Question:', example['question'])
print('Reference answer:', example['reference_answer'])
print('Review status:', example['review_status'])
table(example['spans'], ['source_document', 'start_char', 'end_char', 'text'])
print('Selection rationale:', example['rationale'])


Question: Marco transfers cryptocurrency through multiple wallets to make it harder to trace, fully aware it came from a fraud scheme. Under the Victorian “dealing with proceeds of crime” offences, which physical element is clearly satisfied by his conduct, and which highest mental state could be charged on these facts?
Reference answer: Marco’s conduct is best characterised as dealing with proceeds of crime with intent to conceal that it is proceeds of crime under s 194(1). As the bench book explains, to “deal with” property includes “receive, possess, conceal or dispose of” it, and s 194(1) specifically covers cases where the accused intends to conceal that it is proceeds of crime.
Review status: assistant_reviewed_pending_human
Selection rationale: The question requests both physical conduct and mental state, supported by two separate clauses in the same passage.


source_document,start_char,end_char,text
7.5.22-c1,623,699,intends to conceal that it is proceeds of crime (*Crimes Act 1958* s 194(1))
7.5.22-c1,1033,1094,"“deal with” includes receive, possess, conceal or dispose of."


## 3. One evidence target, three chunk sizes

Parents and children retain their existing boundaries. Grandchildren split each child into at most 128 WordPiece tokens with approximately 32-token overlap. Every retrieval unit points back to the same canonical source. No gold annotation is used to choose chunk boundaries.

The tokenizer is used for counting only, with padding and truncation disabled.

In [3]:
table(csv_rows('corpus_profile.csv'), ['mode', 'units', 'mean_tokens', 'min_tokens', 'max_tokens'])

mode,units,mean_tokens,min_tokens,max_tokens
parent,2785,505.8157989228007,17,12632
child,4876,288.90422477440524,3,576
grandchild,15617,112.21995261573926,3,128


## 4. Evaluation independent of chunk boundaries

Union the actual returned source ranges, then ask whether they contain every annotated span. Overlap and duplicates cannot inflate coverage. Two adjacent fragments can jointly cover a span.

- **All spans included:** all required annotated characters are present.
- **Evidence token coverage:** proportion of gold WordPiece tokens fully covered.
- **Whole gold child included:** the older, coarser requirement, shown separately.

The following is a controlled evaluator check, **not retrieval performance**.

In [4]:
table(bench.read_json(ROOT / 'results/boundary_controls.json'), ['case', 'all_spans_included', 'evidence_char_coverage', 'evidence_token_coverage'])

case,all_spans_included,evidence_char_coverage,evidence_token_coverage
one_piece,1,1.0,1.0
two_pieces,1,1.0,1.0
duplicate,1,1.0,1.0
missing_tail,0,0.5,0.47058823529411764


## 5. Actual BM25 retrieval experiment

Index every source unit, not just gold-bearing sources. Retrieve with the unchanged question text only. Fixed BM25 settings: k1=1.5, b=0.75, lowercase word tokenization, positive Robertson IDF. No tuning or answer-based retrieval.

Evaluate top 1/5/10 with no budget and a 2,048-token prefix-packed body budget. A capped parent contributes only its actually returned coordinates. Corpus changes affect ranking as well as context size; this is not a pure boundary-only ablation.

In [5]:
RERUN_BM25 = False
if RERUN_BM25:
    bench.run(ROOT)
    from audit_results import audit
    audit(ROOT)
summary = csv_rows('comparison.csv')
selected = [r for r in summary if r['k'] == '5' and r['budget'] == '2048']
table([{ 'Corpus': r['mode'], 'All spans included': f"{float(r['all_spans_included']):.1%}",
         'Evidence token coverage': f"{float(r['evidence_token_coverage']):.1%}",
         'Whole gold child': f"{float(r['whole_gold_child_included']):.1%}",
         'Mean returned tokens': round(float(r['returned_tokens']), 1)} for r in selected])


Corpus,All spans included,Evidence token coverage,Whole gold child,Mean returned tokens
parent,35.0%,35.0%,35.0%,2038.1
child,40.0%,40.0%,40.0%,1834.7
grandchild,10.0%,24.9%,0.0%,621.1


## 6. What the draft-label results show

At k=5 and a 2,048-token cap, full proposed-evidence inclusion is 35% for parents, 40% for children and 10% for grandchildren. Grandchildren include full proposed evidence for two questions even though they never include a whole original gold child.

This shows the usefulness of finer evidence labels—not that small chunks win. Five grandchildren return much less text. Parent inclusion falls from 45% without a cap to 35% with the cap. These are 20-question draft-label development results, not the earlier held-out Qwen scores.

In [6]:
table([r for r in summary if r['k'] in ['5','10']],
      ['mode','k','budget','all_spans_included','evidence_token_coverage','returned_tokens'])
rows = csv_rows('per_query.csv')
examples = [r for r in rows if r['mode']=='grandchild' and r['k']=='5' and r['budget']=='2048'
            and float(r['all_spans_included'])==1 and float(r['whole_gold_child_included'])==0]
print('Actual queries where fine evidence is complete but the original gold child is not:')
for row in examples:
    a = next(a for a in annotations if a['question_id']==row['question_id'])
    print('Q' + a['question_id'] + ': ' + a['question'])


Actual queries where fine evidence is complete but the original gold child is not:
Q91: Carmen submits a valuation certificate for 500 crates of “premium truffles” that don’t exist and uses it to support a finance application. Which case has been treated as authority that a valuation of non-existent goods is fraudulent in this way?
Q98: “The salt of the abovementioned drugs, derivates and _______” Which word goes in the blank and which act does this phrase belong to?


mode,k,budget,all_spans_included,evidence_token_coverage,returned_tokens
parent,5,unbounded,0.45,0.45,6147.15
parent,5,2048,0.35,0.35,2038.1
parent,10,unbounded,0.5,0.5,10976.2
parent,10,2048,0.35,0.35,2048.0
child,5,unbounded,0.4,0.4,1894.7
child,5,2048,0.4,0.4,1834.7
child,10,unbounded,0.4,0.4,3706.0
child,10,2048,0.4,0.4,2048.0
grandchild,5,unbounded,0.1,0.24911282691097872,621.1
grandchild,5,2048,0.1,0.24911282691097872,621.1


## 7. Verification and remaining review

The tests validate source coordinates, boundary merging, duplicate handling, budget truncation, Unicode offsets and BM25 arithmetic. The independent audit recomputes coverage using character sets. Neither replaces human review of the chosen evidence.

Priorities: review the 20 cases, adjudicate Q48's possibility/probability mismatch, document acceptable alternative evidence, then let teammates apply the evaluator to the same approved question set. Multiple clauses in one passage are not proof of multi-document completeness.

In [7]:
table([bench.read_json(ROOT / 'results/audit.json')])
tests = bench.read_json(ROOT / 'results/test_report.json')
table([{key: tests[key] for key in ['tests_run', 'failures', 'errors']}])
print('Review sheet:', 'REVIEW_SHEET.md')
print('Teammate interface:', 'TEAM_HANDOFF.md')


Review sheet: REVIEW_SHEET.md
Teammate interface: TEAM_HANDOFF.md


query_setting_rows_independently_checked,summary_rows_checked,character_union_token_coverage_and_budget_checks,human_semantic_review
360,18,passed,not performed; cannot be replaced by these tests


tests_run,failures,errors
22,0,0
